# 00 · Data Ingestion

Collects all raw data into the Bronze layer.

| # | Source | Content | Method | Output |
|---|--------|---------|--------|--------|
| A | Kaggle `lylebegbie` | Tier 1 matches 1995–2024 | `kagglehub` (automatic) | `bronze_kaggle.parquet` |
| B | Flashscore | All RWC matches 1987–2023 | Selenium scrape | `bronze_rwc_matches.parquet` |
| C | World Rugby Rankings | Monthly ratings 2003–2024 | Selenium scrape | `bronze_rankings.parquet` |
| D | RWC stage history | Pedigree features per team | derived from Part B | `bronze_rwc_history.parquet` |

**All parts run automatically — no manual file preparation needed.**  

**Citations:**  
Begbie, L. (2022). *International Rugby Union Results*. Kaggle. https://doi.org/10.34740/KAGGLE/DSN/2510185  
Flashscore (2024). *Rugby World Cup Results*. https://www.flashscore.com/rugby-union/world/  
World Rugby (2024). *Men's Rankings*. https://www.world.rugby/rankings  
World Rugby (2024). *RWC History*. https://www.world.rugby/tournaments/rugbyworldcup/history

In [1]:
# Setup
from pathlib import Path
import time, re, json as _json
import pandas as pd
import numpy as np
import kagglehub

ROOT       = Path('..').resolve()
BRONZE_DIR = Path('../data/bronze')
BRONZE_DIR.mkdir(parents=True, exist_ok=True)
MIN_YEAR   = 1995  # professional era only
MAX_WR_YEAR = 2024  # collect WR rankings up to end of 2024
print(f'Bronze dir: {BRONZE_DIR.resolve()}')

Bronze dir: /Users/tamara.brand/Desktop/springboks-rugby-analytics/data/bronze


## A — Kaggle: Tier 1 Match Results 1995–2024

Full match history for the 10 Tier 1 nations. Downloaded automatically.

In [2]:
dataset_path = Path(kagglehub.dataset_download(
    'lylebegbie/international-rugby-union-results-from-18712022'
))
results_file = next(p for p in dataset_path.rglob('*.csv') if 'results' in p.name.lower())

df_k = pd.read_csv(results_file)
df_k['date']       = pd.to_datetime(df_k['date'], errors='coerce')
df_k['home_score'] = pd.to_numeric(df_k['home_score'], errors='coerce')
df_k['away_score'] = pd.to_numeric(df_k['away_score'], errors='coerce')
df_k = (df_k
        .rename(columns={'competition': 'tournament'})
        .loc[df_k['date'].dt.year >= MIN_YEAR]
        .dropna(subset=['date', 'home_score', 'away_score'])
        .drop_duplicates(subset=['date', 'home_team', 'away_team'])
        .sort_values('date').reset_index(drop=True))

df_k.to_parquet(BRONZE_DIR / 'bronze_kaggle.parquet', index=False)
print(f'A passed Kaggle: {len(df_k):,} matches | '
      f'{df_k.date.min().date()} – {df_k.date.max().date()}')
print(f'   Teams: {sorted(set(df_k.home_team) | set(df_k.away_team))}')

A passed Kaggle: 1,396 matches | 1995-01-21 – 2024-08-17
   Teams: ['Argentina', 'Australia', 'England', 'France', 'Ireland', 'Italy', 'New Zealand', 'Scotland', 'South Africa', 'Wales']


## B — Flashscore: All RWC Match Results 1987–2023

Scrapes all pool + knockout match results for every Rugby World Cup  
from Flashscore using Selenium. Covers all Tier 2 nations.  

**Runtime:** ~3–5 min (10 tournament pages)

In [3]:
from selenium import webdriver
from selenium.webdriver.edge.service import Service
from selenium.webdriver.edge.options import Options
from webdriver_manager.microsoft import EdgeChromiumDriverManager

RWC_URLS = {
    1987: 'https://www.flashscore.com/rugby-union/world/world-cup-1987/results/',
    1991: 'https://www.flashscore.com/rugby-union/world/world-cup-1991/results/',
    1995: 'https://www.flashscore.com/rugby-union/world/world-cup-1995/results/',
    1999: 'https://www.flashscore.com/rugby-union/world/world-cup-1999/results/',
    2003: 'https://www.flashscore.com/rugby-union/world/world-cup-2003/results/',
    2007: 'https://www.flashscore.com/rugby-union/world/world-cup-2007/results/',
    2011: 'https://www.flashscore.com/rugby-union/world/world-cup-2011/results/',
    2015: 'https://www.flashscore.com/rugby-union/world/world-cup-2015/results/',
    2019: 'https://www.flashscore.com/rugby-union/world/world-cup-2019/results/',
    2023: 'https://www.flashscore.com/rugby-union/world/world-cup-2023/results/',
}

# ── launch Edge ───────────────────────────────────────────────────────────
fs_opts = Options()
fs_opts.add_argument('--headless=new')
fs_opts.add_argument('--no-sandbox')
fs_opts.add_argument('--disable-dev-shm-usage')
fs_opts.add_argument('--window-size=1400,900')
fs_opts.add_argument('--disable-blink-features=AutomationControlled')
fs_opts.add_experimental_option('excludeSwitches', ['enable-automation'])
fs_opts.add_experimental_option('useAutomationExtension', False)
fs_opts.add_argument(
    '--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
    'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
)

fs_driver = webdriver.Edge(
    service=Service(EdgeChromiumDriverManager().install()),
    options=fs_opts,
)

# ── parse matches from current page ──────────────────────────────────────
PARSE_JS = """
const rows = [];
document.querySelectorAll('[class*="event__match"]').forEach(m => {
    try {
        const home = m.querySelector('.event__homeParticipant')?.innerText.trim();
        const away = m.querySelector('.event__awayParticipant')?.innerText.trim();
        const date = m.querySelector('.event__time')?.innerText.trim();
        // scores: last two bold spans with numeric content
        const spans = Array.from(m.querySelectorAll('span[class*="wcl-bold"]'))
            .map(s => s.innerText.trim())
            .filter(t => /^\\d+$/.test(t));
        if (home && away && spans.length >= 2) {
            rows.push({
                home_team:  home,
                away_team:  away,
                home_score: parseInt(spans[spans.length-2]),
                away_score: parseInt(spans[spans.length-1]),
                date:       date || '',
            });
        }
    } catch(e) {}
});
return rows;
"""

# ── scrape all tournaments ────────────────────────────────────────────────
all_matches = []
for year, url in RWC_URLS.items():
    fs_driver.get(url)
    time.sleep(3)

    # close overlays via JS
    fs_driver.execute_script("""
        document.querySelectorAll(
            '[class*="close"], [class*="consent"], [id*="onetrust-accept"]'
        ).forEach(b => { try { b.click(); } catch(e) {} });
    """)
    time.sleep(1)

    # scroll to load all matches
    fs_driver.execute_script('window.scrollTo(0, document.body.scrollHeight)')
    time.sleep(2)
    fs_driver.execute_script('window.scrollTo(0, 0)')
    time.sleep(1)

    matches = fs_driver.execute_script(PARSE_JS)
    for m in matches:
        m['tournament'] = f'Rugby World Cup {year}'
        m['year']       = year
        m['neutral']    = True
        all_matches.append(m)
    print(f'  {year}: {len(matches):2d} matches scraped')

fs_driver.quit()

# ── parse dates + save ────────────────────────────────────────────────────
df_rwc_m = pd.DataFrame(all_matches)
# flashscore dates: 'DD.MM.YYYY' format
df_rwc_m['date'] = pd.to_datetime(df_rwc_m['date'], dayfirst=True, errors='coerce')
# fill missing dates with Oct 1 of the tournament year
df_rwc_m['date'] = df_rwc_m.apply(
    lambda r: r['date'] if pd.notna(r['date'])
    else pd.Timestamp(f"{r['year']}-10-01"), axis=1
)
df_rwc_m = df_rwc_m.sort_values('date').reset_index(drop=True)
df_rwc_m.to_parquet(BRONZE_DIR / 'bronze_rwc_matches.parquet', index=False)

all_teams = set(df_rwc_m.home_team) | set(df_rwc_m.away_team)
print(f'\nB passed Flashscore RWC: {len(df_rwc_m):,} matches | '
      f'{len(all_teams)} teams')
print('   Tier 2 spot-check:')
for t in ['Georgia', 'Romania', 'Fiji', 'Uruguay', 'Chile', 'Zimbabwe']:
    n = len(df_rwc_m[(df_rwc_m.home_team==t)|(df_rwc_m.away_team==t)])
    print(f'   {"passed" if n>0 else "failed"} {t}: {n} matches')


  1987: 32 matches scraped


  1991: 32 matches scraped


  1995: 32 matches scraped


KeyboardInterrupt: 

## C — World Rugby Rankings: Weekly Snapshots 2003–2024

Scrapes all available weekly Men's Rankings from the official World Rugby  
website using Selenium. Navigates the date picker for every available date.  

**Runtime:** ~15–25 min (~1,100 weekly snapshots × ~50 teams)

In [ ]:
from selenium import webdriver
from selenium.webdriver.edge.service import Service
from selenium.webdriver.edge.options import Options
from webdriver_manager.microsoft import EdgeChromiumDriverManager

opts = Options()
opts.add_argument('--headless=new')
opts.add_argument('--no-sandbox')
opts.add_argument('--disable-dev-shm-usage')
opts.add_argument('--window-size=1400,900')
opts.add_argument('--disable-blink-features=AutomationControlled')
opts.add_experimental_option('excludeSwitches', ['enable-automation'])
opts.add_experimental_option('useAutomationExtension', False)
opts.add_argument(
    '--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
    'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
)

driver = webdriver.Edge(
    service=Service(EdgeChromiumDriverManager().install()),
    options=opts,
)
driver.get('https://www.world.rugby/rankings')
time.sleep(5)
print(f'C — Edge launched | page: {driver.title}')

In [ ]:
# ── collect all available dates by iterating year dropdown ───────────────
JS_GET_YEARS = """
    return Array.from(document.querySelectorAll('.js-year-button .js-date-option'))
        .map(o => ({value: o.dataset.value,
                    year:  parseInt((o.dataset.value||'').substring(0,4))}))
        .filter(o => o.year && o.year <= 2024);
"""
JS_GET_MONTHS = """
    return Array.from(document.querySelectorAll(
        '.wr-datepicker__button:not(.js-year-button) .js-date-option'
    )).map(o => o.dataset.value).filter(Boolean);
"""
JS_SHOW_FULL = """
    const btn = Array.from(document.querySelectorAll('button,span'))
        .find(b => b.innerText && b.innerText.includes('SHOW FULL'));
    if (btn) btn.click();
"""

def click_year(year_value):
    js = (
        "const yearBtn = document.querySelector('.js-year-button .js-date-select');"
        "if (!yearBtn) return 'no_btn';"
        "yearBtn.click();"
        "const opts = Array.from(document.querySelectorAll('.js-year-button .js-date-option'));"
        f"const opt = opts.find(o => o.dataset.value === '{year_value}');"
        "if (!opt) return 'no_opt';"
        "opt.click(); return 'ok';"
    )
    return driver.execute_script(js) == 'ok'

# expand full table
driver.execute_script(JS_SHOW_FULL)
time.sleep(1)

year_options = driver.execute_script(JS_GET_YEARS)
all_dates    = []
print(f'Years available (<=2024): {len(year_options)}')

for yo in year_options:
    ok = click_year(yo['value'])
    time.sleep(1.2)
    if ok:
        dates = [d for d in driver.execute_script(JS_GET_MONTHS)
                 if d and d <= '2024-12-31']
        all_dates.extend(dates)
        print(f'  {yo["year"]}: {len(dates)} dates')
    else:
        print(f'  {yo["year"]}: click failed')

# ── filter to monthly snapshots (first available date per month) ─────
all_dates_weekly  = sorted(set(all_dates))
seen_months, all_dates = set(), []
for d in all_dates_weekly:
    month_key = d[:7]  # 'YYYY-MM'
    if month_key not in seen_months:
        seen_months.add(month_key)
        all_dates.append(d)

print(f'\nWeekly snapshots available: {len(all_dates_weekly)}')
print(f'Monthly snapshots to scrape: {len(all_dates)} (~4x faster)')
print(f'Range: {all_dates[0]} – {all_dates[-1]}')


In [ ]:
def click_date(date_value):
    year = date_value[:4]
    r = driver.execute_script(
        "const b = document.querySelector('.js-year-button .js-date-select');"
        "if (!b) return 'no_year';"
        "b.click();"
        "const opts = Array.from(document.querySelectorAll('.js-year-button .js-date-option'));"
        f"const opt = opts.find(o => o.dataset.value && o.dataset.value.startsWith('{year}-'));"
        "if (!opt) return 'no_year_opt';"
        "opt.click(); return 'ok';"
    )
    if r != 'ok': return False
    time.sleep(0.8)
    r2 = driver.execute_script(
        "const b = document.querySelector("
        "  '.wr-datepicker__button:not(.js-year-button) .js-date-select');"
        "if (!b) return 'no_month';"
        "b.click();"
        "const opts = Array.from(document.querySelectorAll("
        "  '.wr-datepicker__button:not(.js-year-button) .js-date-option'));"
        f"const opt = opts.find(o => o.dataset.value === '{date_value}');"
        "if (!opt) return 'no_date_opt';"
        "opt.click(); return 'ok';"
    )
    return r2 == 'ok'

def parse_mens():
    raw = driver.execute_script(
        "const c = document.querySelector('[class*=\"rankings\"]');"
        "return c ? c.innerText : '';"
    )
    if not raw: return []
    pos_idx  = raw.find('\nPOS\n')
    mens_idx = raw.find("MEN'S", pos_idx)
    if mens_idx < 0: return []
    lines = [l.strip() for l in raw[mens_idx:].split('\n') if l.strip()]
    SKIP  = {"MEN'S",'TEAMS','POINTS','POS','SHOW FULL TABLE',
              'RANKINGS EXPLAINED','BIGGEST CLIMBERS','BIGGEST FALLERS','CHANGE'}
    teams = []
    for i, line in enumerate(lines):
        if len(teams) >= 60: break
        if re.match(r'^\d{2,3}\.\d{1,2}$', line):
            j = i - 1
            while j >= 0 and (
                not lines[j] or lines[j] in SKIP
                or lines[j].isdigit()
                or re.match(r'^\(\d+\)$', lines[j])
                or re.match(r'^[+-]\d+$', lines[j])
            ):
                j -= 1
            if j >= 0 and lines[j] not in SKIP and len(lines[j]) > 1:
                teams.append({'rank': len(teams)+1,
                              'team': lines[j],
                              'rating_pts': float(line)})
    return teams

def launch_driver():
    """Launch a fresh Edge driver."""
    d = webdriver.Edge(
        service=Service(EdgeChromiumDriverManager().install()),
        options=opts,
    )
    d.get('https://www.world.rugby/rankings')
    time.sleep(5)
    d.execute_script(JS_SHOW_FULL)
    time.sleep(1)
    return d

# ── main scrape loop ──────────────────────────────────────────────────────
all_rows, n_failed = [], 0

# re-launch driver (previous cells may have closed it)
try:
    driver.title  # test if still alive
except Exception:
    print('Driver closed — relaunching...')
    driver = launch_driver()

for i, date_val in enumerate(all_dates):
    # auto-restart if driver crashed
    try:
        driver.title
    except Exception:
        print(f'  Driver crashed at {date_val} — relaunching...')
        driver = launch_driver()

    ok = click_date(date_val)
    if not ok:
        n_failed += 1
        continue

    time.sleep(1.2)
    driver.execute_script(JS_SHOW_FULL)
    time.sleep(0.5)

    teams = parse_mens()
    if teams:
        for t in teams:
            all_rows.append({'date': date_val, 'rank': t['rank'],
                             'team': t['team'], 'rating_pts': t['rating_pts']})
        if (i + 1) % 50 == 0 or i == 0:
            sa     = next((t for t in teams if t['team'] == 'South Africa'), None)
            sa_str = f"SA #{sa['rank']} ({sa['rating_pts']})" if sa else 'SA?'
            print(f'  [{i+1:4d}/{len(all_dates)}] {date_val} | {len(teams)} teams | {sa_str}')
    else:
        n_failed += 1

try:
    driver.quit()
except Exception:
    pass
print(f'\nDone: {len(all_rows):,} rows | '
      f'{len(set(r["date"] for r in all_rows))} snapshots | '
      f'{n_failed} failed')


In [ ]:
if not all_rows:
    print('ERROR: no WR data collected')
else:
    df_wr = (pd.DataFrame(all_rows)
             .assign(date=lambda d: pd.to_datetime(d.date))
             .drop_duplicates(subset=['date', 'team'])
             .sort_values(['date', 'rank'])
             .reset_index(drop=True))
    df_wr.to_parquet(BRONZE_DIR / 'bronze_rankings.parquet', index=False)
    print(f'C WR Rankings: {len(df_wr):,} rows | '
          f'{df_wr.date.nunique()} snapshots | '
          f'{df_wr.date.min().date()} – {df_wr.date.max().date()}')
    print('\nSouth Africa (every 52nd ≈ annual):')
    sa = df_wr[df_wr.team == 'South Africa'][['date','rank','rating_pts']]
    print(sa.iloc[::52].to_string(index=False))

## D — RWC Stage History *(derived from Flashscore)*

Computes cumulative pedigree features per team per RWC tournament  
directly from the match data scraped in Part B.  
No hardcoding — fully derived from source data.  
Stage encoding inferred from match section labels: `pool=1`, `qf=2`, `sf=3`, `final=4`, `winner=5`

In [ ]:
# ── D: RWC Stage History — loaded from sources/rwc_stage_history.csv ──────
# Source: World Rugby (2024). RWC History.
# https://www.world.rugby/tournaments/rugbyworldcup/history
# Data manually compiled and stored as CSV for reproducibility.
# Stage encoding: pool=1, qf=2, sf=3, final=4, winner=5

sources_dir = ROOT / 'data' / 'sources'
assert (sources_dir / 'rwc_stage_history.csv').exists(), (
    'Missing: data/sources/rwc_stage_history.csv\n'
    'Copy from the sources/ folder in the project repository.'
)

df_rwc = pd.read_csv(sources_dir / 'rwc_stage_history.csv')
df_rwc = df_rwc.rename(columns={'stage_code': 'stage'})

# compute cumulative pedigree — strictly before each tournament (no leakage)
rows = []
for team, grp in df_rwc.sort_values(['team','rwc_year']).groupby('team'):
    for _, r in grp.iterrows():
        past = grp[grp.rwc_year < r.rwc_year]
        rows.append({
            'team':            team,
            'rwc_year':        r.rwc_year,
            'rwc_stage':       r.stage,
            'rwc_appearances': len(past),
            'rwc_best_stage':  int(past.stage.max()) if len(past) else 0,
            'rwc_cumul_score': int(past.stage.sum()) if len(past) else 0,
        })

df_rwc_feat = pd.DataFrame(rows)
df_rwc_feat.to_parquet(BRONZE_DIR / 'bronze_rwc_history.parquet', index=False)

print(f'D - RWC history: {len(df_rwc_feat)} entries | {df_rwc_feat.team.nunique()} teams')
print('   Loaded from: data/sources/rwc_stage_history.csv')
print('   South Africa pedigree:')
print(df_rwc_feat[df_rwc_feat.team == 'South Africa']
      [['rwc_year','rwc_stage','rwc_appearances','rwc_best_stage','rwc_cumul_score']]
      .to_string(index=False))


In [ ]:
print('\n' + '='*55)
print('BRONZE LAYER SUMMARY')
print('='*55)
for fname, label in [
    ('bronze_kaggle.parquet',      'A  Kaggle — Tier 1 matches'),
    ('bronze_rwc_matches.parquet', 'B  Flashscore — RWC matches'),
    ('bronze_rankings.parquet',    'C  WR Rankings (weekly)'),
    ('bronze_rwc_history.parquet', 'D  RWC stage history'),
]:
    path = BRONZE_DIR / fname
    if path.exists():
        n = len(pd.read_parquet(path))
        print(f' passed {label}: {n:,} rows')
    else:
        print(f'  failed {label}: missing')
print('='*55)
print('Next: run 01_data_cleaning.ipynb')